# 使用 CLIP 嵌入和 GPT-4 Vision 实现多模态 RAG


多模态 RAG 将更多模态整合到传统的基于文本的 RAG 中，通过提供额外的上下文并为文本数据提供依据（grounding）来增强理解，从而提升 LLM 的问答能力。

我们沿用 [服装搭配 cookbook](https://cookbook.openai.com/examples/how_to_combine_gpt4v_with_rag_outfit_assistant) 中的方法，直接对图像进行嵌入以做相似度搜索，绕过有信息损失的文本描述生成过程，从而提升检索准确率。

使用基于 CLIP 的嵌入还可以利用特定数据进行微调，或用未见过的图像进行更新。

我们将通过一个示例来展示这一技术：使用用户提供的科技产品图像搜索企业知识库，从而提供相关信息。

# 安装

首先安装相关的包。

In [ ]:
#installations
%pip install clip
%pip install torch
%pip install pillow
%pip install faiss-cpu
%pip install numpy
%pip install git+https://github.com/openai/CLIP.git
%pip install openai

然后导入所有需要的包。


In [3]:
# model imports
import faiss
import json
import torch
from openai import OpenAI
import torch.nn as nn
from torch.utils.data import DataLoader
import clip
client = OpenAI()

# helper imports
from tqdm import tqdm
import json
import os
import numpy as np
import pickle
from typing import List, Union, Tuple

# visualisation imports
from PIL import Image
import matplotlib.pyplot as plt
import base64

现在加载 CLIP 模型。

In [4]:
#load model on device. The device you are running inference/training on is either a CPU or GPU if you have.
device = "cpu"
model, preprocess = clip.load("ViT-B/32",device=device)


接下来我们将：
1.   创建图像嵌入数据库
2.   设置对视觉模型的查询
3.   执行语义搜索
4.   将用户查询连同图像一起传入



# 创建图像嵌入数据库

接下来，我们将基于一个图像目录创建图像嵌入知识库。这就是我们要搜索的科技产品知识库，用于针对用户上传的图像向其提供信息。

我们传入存放图像（JPEG 格式）的目录，并遍历每张图像来创建嵌入。

我们还有一个 description.json。它为知识库中的每一张图像都提供了一个条目，包含两个键：'image_path' 和 'description'。它将每张图像映射到一段有用的描述，以帮助回答用户的问题。

首先编写一个函数，获取给定目录中的所有图像路径。然后我们将从名为 'image_database' 的目录中获取所有 jpeg 图像

In [5]:
def get_image_paths(directory: str, number: int = None) -> List[str]:
    image_paths = []
    count = 0
    for filename in os.listdir(directory):
        if filename.endswith('.jpeg'):
            image_paths.append(os.path.join(directory, filename))
            if number is not None and count == number:
                return [image_paths[-1]]
            count += 1
    return image_paths
direc = 'image_database/'
image_paths = get_image_paths(direc)


接下来编写一个函数，在给定一系列路径的情况下，从 CLIP 模型获取图像嵌入。

我们首先使用之前得到的 preprocess 函数对图像进行预处理。它会执行若干操作，确保 CLIP 模型的输入具有正确的格式和维度，包括调整尺寸、归一化、颜色通道调整等。

然后将这些预处理后的图像堆叠在一起，这样就可以一次性传入模型，而不必循环处理。最后返回模型输出，即一个嵌入数组。

In [6]:
def get_features_from_image_path(image_paths):
  images = [preprocess(Image.open(image_path).convert("RGB")) for image_path in image_paths]
  image_input = torch.tensor(np.stack(images))
  with torch.no_grad():
    image_features = model.encode_image(image_input).float()
  return image_features
image_features = get_features_from_image_path(image_paths)


现在我们可以创建向量数据库了。

In [7]:
index = faiss.IndexFlatIP(image_features.shape[1])
index.add(image_features)


同时读取用于图像-描述映射的 json，并创建一个 json 列表。我们还创建了一个辅助函数，用于在该列表中查找指定的图像，从而获取该图像的描述

In [11]:
data = []
image_path = 'train1.jpeg'
with open('description.json', 'r') as file:
    for line in file:
        data.append(json.loads(line))
def find_entry(data, key, value):
    for entry in data:
        if entry.get(key) == value:
            return entry
    return None

我们来展示一张示例图像，它将作为用户上传的图像。这是一款在 2024 年 CES 上发布的科技产品：DELTA Pro Ultra 全屋电池发电机。

In [ ]:
im = Image.open(image_path)
plt.imshow(im)
plt.show()

![Delta Pro](../images/train1.jpeg)

# 查询视觉模型

现在来看看 GPT-4 Vision（它之前不可能见过这款产品）会给它打上什么标签。



首先需要编写一个函数，将图像编码为 base64，因为这是传入视觉模型的格式。然后创建一个通用的 image_query 函数，以便用图像输入查询 LLM。

In [12]:
def encode_image(image_path):
    with open(image_path, 'rb') as image_file:
        encoded_image = base64.b64encode(image_file.read())
        return encoded_image.decode('utf-8')

def image_query(query, image_path):
    response = client.chat.completions.create(
        model='gpt-4-vision-preview',
        messages=[
            {
            "role": "user",
            "content": [
                {
                "type": "text",
                "text": query,
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{encode_image(image_path)}",
                },
                }
            ],
            }
        ],
        max_tokens=300,
    )
    # Extract relevant features from the response
    return response.choices[0].message.content
image_query('Write a short label of what is show in this image?', image_path)

'Autonomous Delivery Robot'

可以看到，它根据训练所学的信息尽力作答，但由于训练数据中没有见过类似的东西，它犯了错误。这是因为这是一张有歧义的图像，很难据此推断和推理。

# 执行语义搜索

现在执行相似度搜索，在知识库中找出最相似的两张图像。具体做法是对用户提供的 `image_path` 进行嵌入，然后检索数据库中图像的索引和相似度分数。分数越大表示越相似，因此我们按降序对分数排序。

In [13]:
image_search_embedding = get_features_from_image_path([image_path])
distances, indices = index.search(image_search_embedding.reshape(1, -1), 2) #2 signifies the number of topmost similar images to bring back
distances = distances[0]
indices = indices[0]
indices_distances = list(zip(indices, distances))
indices_distances.sort(key=lambda x: x[1], reverse=True)

我们使用这些索引在 `image_directory` 中查找，并选出对应的图像传给视觉模型用于 RAG。

我们来看看它返回了什么（按相似度顺序展示）：

In [ ]:
#display similar images
for idx, distance in indices_distances:
    print(idx)
    path = get_image_paths(direc, idx)[0]
    im = Image.open(path)
    plt.imshow(im)
    plt.show()

![Delta Pro2](../images/train2.jpeg)

![Delta Pro3](../images/train17.jpeg)

可以看到，它返回了两张包含 DELTA Pro Ultra 全屋电池发电机的图像。其中一张图像还带有一些可能造成干扰的背景，但它仍然成功找到了正确的图像。

# 用户查询最相似的图像

现在，对于最相似的图像，我们希望将它及其描述连同用户查询一起传给 gpt-v，以便用户询问他们可能购买的这款产品。这正是视觉模型威力的体现：你可以向模型提出它没有专门训练过的一般性问题，而它能以很高的准确率作答。

在下面的示例中，我们将询问该产品的容量。

In [14]:
similar_path = get_image_paths(direc, indices_distances[0][0])[0]
element = find_entry(data, 'image_path', similar_path)

user_query = 'What is the capacity of this item?'
prompt = f"""
Below is a user query, I want you to answer the query using the description and image provided.

user query:
{user_query}

description:
{element['description']}
"""
image_query(prompt, similar_path)

'The portable home battery DELTA Pro has a base capacity of 3.6kWh. This capacity can be expanded up to 25kWh with additional batteries. The image showcases the DELTA Pro, which has an impressive 3600W power capacity for AC output as well.'

可以看到它能够回答这个问题。这之所以可行，完全是因为我们直接匹配了图像，并由此收集到相关描述作为上下文。

# 总结

在本 notebook 中，我们介绍了如何使用 CLIP 模型，给出了使用 CLIP 模型创建图像嵌入数据库的示例，执行了语义搜索，最后提供用户查询来回答问题。

这种使用模式可以应用于许多不同的领域，而且很容易进一步改进这一技术。例如，你可以微调 CLIP，可以像在 RAG 中那样改进检索过程，还可以对 GPT-V 进行提示工程。
